TODO: natively vectorize as many functions as possible to avoid iterations and speed up computation.

In [ ]:
from engine import WordleEngine
import numpy as np
from constants import Square
from wordfreq import zipf_frequency


WORD_LEN = 5
LOCALE = 'en'
FREQ_VEC = np.vectorize(lambda word: zipf_frequency(word, LOCALE), otypes=[float])

engine = WordleEngine('words.txt', WORD_LEN)
df = engine._df
patterns = engine._patterns

In [ ]:
def convert_str_to_squares(abbrev: str) -> np.ndarray:
    """Convenience wrapper to save myself some copy pasting."""
    squares = []
    for letter in abbrev:
        assert letter in ('g', 'y', 'b')
        if letter == 'g':
            squares.append(Square.GREEN.value)
        elif letter == 'y':
            squares.append(Square.YELLOW.value)
        else:
            squares.append(Square.BLACK.value)
    return np.array(squares, dtype=np.uint8)

In [ ]:
import pandas as pd
from cache import _is_match

def filter_results(current: pd.DataFrame, guess: str, response: str) -> np.ndarray:
    squares = convert_str_to_squares(response)
    gs_np = df.loc[guess].to_numpy()
    guess_idx = df.index.get_loc(guess)
    mask = [_is_match(gs_np, row.to_numpy(), squares) \
            for _, row in current.iterrows()]
    mask2 = []
    for ind, _ in current.iterrows():
        cand_idx = df.index.get_loc(ind)
        assert patterns is not None
        match = np.all(patterns[guess_idx, cand_idx, :] == squares)
        mask2.append(match)
    return np.array(mask)

In [ ]:
from scipy.stats import entropy

def get_entropy(current: np.ndarray):
    entrops = np.empty((current.shape[0],), dtype=float)
    for idx in range(current.shape[0]):
        _, counts = np.unique(current[idx, :, :], axis=0, return_counts=True)
        entrops[idx] = entropy(counts / current.shape[1], base=2)
    return entrops

Play a game of Worlde below. Record your guesses and responses as you make them and rerun the following cells to refresh the suggestions.

In [ ]:
guesses = [
    'tares', # the optimal starting word
    'nitty',
    'halwa',
    'watch',
]
responses = [
    'ygbbb',
    'bbgbb',
    'ygbyb',
    'ggggg',
]

Iteratively update both suggested solutions and most informative guesses lists.

In [ ]:
from math import log2
filtered = df.copy()
suggest = df.copy()
assert patterns is not None
patts = np.copy(patterns)

print(f'Starting entropy = {round(log2(len(filtered)), 3)}')
for guess, response in zip(guesses, responses):
    mask = filter_results(filtered, guess, response)
    filtered = filtered[mask]
    print(mask)
    print(np.count_nonzero(mask))
    entropy_lost = -log2(np.count_nonzero(mask) / len(mask))
    print(f'\tLost entropy = {round(entropy_lost, 3)}')
    print(f'Remaining entropy = {round(log2(len(filtered)), 3)}')
    patts = patts[:, mask, :]

recc = filtered.copy()
recc['freq'] = FREQ_VEC(recc.index)
recc.sort_values(by='freq', ascending=False, inplace=True)
print('Possible solutions sorted by log freq')
print(recc[:10], '=' * 35, sep='\n')

entrops = get_entropy(patts)
suggest['entropy'] = entrops
print('Most likely informative next guesses sorted by entropy')
suggest.sort_values('entropy', ascending=False, inplace=True)
print(suggest[:10], '=' * 35, sep='\n')

Once you have the solution, visually ascertain that the produced squares match the game and validate our own function.

In [ ]:
expected = 'watch'

for guess, response in zip(guesses, responses):
    squares = engine.lookup_pattern(guess, expected)
    print(guess, ''.join(squares))

A bot playing wordle should always seek the highest entropy word unless it's down to 2 or 3 (less than 2 bits entropy). Then pick the more frequent option.